# Reproducibility & Environments - Solutions

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 2/6

Complete solutions with explanations, expected outputs, and best practices notes.

## Solution 1: Dependency Detective

Understanding why models break when dependencies drift.

In [ ]:
# ANSWERS:

# 1. Root cause:
# The model was trained with an older version of scikit-learn, but the teammate
# installed a newer version (allowed by >=). Internal attribute names changed
# between versions (n_features_ was renamed in sklearn 1.0+).

# 2. Missing information:
# - Exact versions that were actually used at training time (requirements.lock or pip freeze output)
# - Python version used for training
# - Better: a complete environment snapshot

# 3. Command:
# pip freeze > requirements.lock
# This captures EVERY installed package with exact versions, including transitive dependencies

# 4. >= vs ==:
# For applications (including ML training pipelines): use == (exact pins)
# Reason: You want byte-for-byte identical behavior. Different versions = different results.
# 
# For libraries that others install: use >= with upper bounds
# Reason: Your library needs to coexist with other packages in the user's environment.

print("Key lesson: 'Works on my machine' = unpinned dependencies")
print("Fix: requirements.lock with EXACT versions for every application")

**Best Practice:** Always commit both `requirements.txt` (human-edited, high-level deps) and `requirements.lock` (machine-generated, complete closure) to version control.

## Solution 2: The Seed Utility

One function to seed them all.

In [ ]:
import os
import random
import numpy as np

def set_all_seeds(seed: int = 42) -> np.random.Generator:
    """Seed all random sources used in classic ML stack."""
    os.environ["PYTHONHASHSEED"] = str(seed)  # for set/dict iteration order (next process)
    random.seed(seed)                         # Python's built-in random module
    np.random.seed(seed)                      # NumPy's legacy global RNG (sklearn uses this)
    return np.random.default_rng(seed)        # Modern Generator for your code

# Test: prove reproducibility
def random_operation():
    """Mix all random sources."""
    a = random.randint(1, 100)
    b = np.random.rand(3)
    c = np.random.randint(1, 100, size=2)
    return a, b, c

# Run 1
rng1 = set_all_seeds(7)
r1_builtin, r1_np_rand, r1_np_int = random_operation()
r1_generator = rng1.integers(0, 100, size=2)

# Run 2 (same seed)
rng2 = set_all_seeds(7)
r2_builtin, r2_np_rand, r2_np_int = random_operation()
r2_generator = rng2.integers(0, 100, size=2)

# Verify
assert r1_builtin == r2_builtin, "Built-in random not seeded!"
assert np.allclose(r1_np_rand, r2_np_rand), "NumPy rand not seeded!"
assert np.array_equal(r1_np_int, r2_np_int), "NumPy randint not seeded!"
assert np.array_equal(r1_generator, r2_generator), "Generator not seeded!"

print("✓ All random sources are reproducible")
print(f"Example output: builtin={r1_builtin}, np_rand={r1_np_rand}, generator={r1_generator}")

**Expected Output:**
```
✓ All random sources are reproducible
Example output: builtin=17, np_rand=[0.77 0.43 0.85], generator=[78 49]
```

**Note:** `PYTHONHASHSEED` affects the *next* process, not the current one. Export it before running: `PYTHONHASHSEED=42 python script.py`

## Solution 3: Reproducible Split Detective

Finding and fixing three seeding bugs.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# BUGS IDENTIFIED:
# Bug 1: train_test_split doesn't receive random_state parameter
# Bug 2: RandomForestClassifier doesn't receive random_state parameter
# Bug 3 (subtle): numpy seed is set once but split happens each execution

# FIXED VERSION:
SEED = 42

def reproducible_training():
    """Fully reproducible training function."""
    np.random.seed(SEED)  # Reset before each run
    
    X, y = load_iris(return_X_y=True)
    
    # FIX 1: Add random_state to train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=True, random_state=SEED
    )
    
    # FIX 2: Add random_state to RandomForestClassifier
    model = RandomForestClassifier(n_estimators=10, random_state=SEED)
    model.fit(X_train, y_train)
    
    return model, X_test, y_test

# Verify reproducibility
model1, X_test1, y_test1 = reproducible_training()
pred1 = model1.predict(X_test1)

model2, X_test2, y_test2 = reproducible_training()
pred2 = model2.predict(X_test2)

assert np.array_equal(X_test1, X_test2), "Test sets differ!"
assert np.array_equal(pred1, pred2), "Predictions differ!"

print("✓ Training is fully reproducible")
print(f"Test accuracy: {model1.score(X_test1, y_test1):.3f}")
print(f"First 5 predictions: {pred1[:5]}")
print(f"\nRun it again - output will be IDENTICAL")

**Key Lesson:** Seeding NumPy's global RNG is necessary but NOT sufficient. Every estimator that uses randomness needs its own `random_state` parameter set explicitly.

## Solution 4: Config-Driven Training

Replacing magic numbers with a typed config object.

In [ ]:
import json
from dataclasses import dataclass, asdict
from pathlib import Path
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib
import numpy as np

@dataclass
class TrainConfig:
    """All hyperparameters in one typed bundle."""
    model_type: str = "RandomForest"
    n_estimators: int = 20
    max_depth: int | None = 5
    seed: int = 42
    test_size: float = 0.2

# Create config
config = TrainConfig(n_estimators=30, max_depth=4, seed=7)
print("Config:", config)

# Train using config values
np.random.seed(config.seed)
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.test_size, random_state=config.seed
)

model = RandomForestClassifier(
    n_estimators=config.n_estimators,
    max_depth=config.max_depth,
    random_state=config.seed
)
model.fit(X_train, y_train)
score = model.score(X_test, y_test)

# Save to run directory
run_dir = Path("sample_runs/config_test")
run_dir.mkdir(parents=True, exist_ok=True)

(run_dir / "config.json").write_text(json.dumps(asdict(config), indent=2))
joblib.dump(model, run_dir / "model.joblib")
(run_dir / "metrics.json").write_text(json.dumps({"accuracy": round(score, 4)}, indent=2))

print(f"\n✓ Saved to {run_dir}/")
print(f"  Accuracy: {score:.3f}")

# Reload and verify
loaded_config = TrainConfig(**json.loads((run_dir / "config.json").read_text()))
loaded_model = joblib.load(run_dir / "model.joblib")
loaded_metrics = json.loads((run_dir / "metrics.json").read_text())

assert loaded_config == config, "Config didn't survive round trip!"
assert np.array_equal(loaded_model.predict(X_test), model.predict(X_test)), "Model changed!"

print("\n✓ Reload successful - config and model match perfectly")
print(f"Loaded config: {loaded_config}")
print(f"Loaded metrics: {loaded_metrics}")

**Best Practice:** The config file becomes the experiment's documentation. Anyone can see exactly what hyperparameters produced what result, without reading code.

## Solution 5: Run Directory Factory

Organizing experiments into self-describing folders.

In [ ]:
import json
from datetime import datetime
from pathlib import Path
import joblib

def create_run_directory(base: str, name: str, timestamp: str | None = None) -> Path:
    """Create timestamped run directory: {base}/{timestamp}_{name}/"""
    root = Path(base)
    ts = timestamp or datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = root / f"{ts}_{name}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir

def save_experiment(run_dir: Path, config: dict, metrics: dict, model=None) -> None:
    """Save all experiment artifacts to run directory."""
    run_dir = Path(run_dir)
    
    # Save config
    (run_dir / "config.json").write_text(json.dumps(config, indent=2))
    
    # Save metrics
    (run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    
    # Save model if provided
    if model is not None:
        joblib.dump(model, run_dir / "model.joblib")
    
    print(f"✓ Saved experiment to {run_dir}/")

# Test with multiple runs
run1 = create_run_directory("sample_runs", "baseline", "20260826_1000")
save_experiment(run1, 
                config={"lr": 0.01, "epochs": 10},
                metrics={"accuracy": 0.85, "loss": 0.34})

run2 = create_run_directory("sample_runs", "tuned", "20260826_1100")
save_experiment(run2,
                config={"lr": 0.001, "epochs": 20},
                metrics={"accuracy": 0.92, "loss": 0.18})

print("\nAll runs:")
for run_dir in sorted(Path("sample_runs").glob("*")):
    if run_dir.is_dir():
        metrics = json.loads((run_dir / "metrics.json").read_text())
        print(f"  {run_dir.name}: accuracy={metrics['accuracy']:.2f}")

**Expected Output:**
```
✓ Saved experiment to sample_runs/20260826_1000_baseline/
✓ Saved experiment to sample_runs/20260826_1100_tuned/

All runs:
  20260826_1000_baseline: accuracy=0.85
  20260826_1100_tuned: accuracy=0.92
```

**Best Practice:** The directory structure itself becomes the experiment log. No database, no special tools—just files that survive decades.

## Solution 6: Content Addressing for Data

Using cryptographic hashes as data version IDs.

In [ ]:
import hashlib
from pathlib import Path

def compute_data_hash(filepath: Path) -> str:
    """Return first 12 chars of SHA-256 hash of file contents."""
    content = Path(filepath).read_bytes()
    return hashlib.sha256(content).hexdigest()[:12]

# Create test files
test_dir = Path("sample_data")
test_dir.mkdir(exist_ok=True)

# Property 1: Same content = same hash
file1 = test_dir / "data_v1.csv"
file1.write_text("id,value\n1,100\n2,200\n3,300\n")

file1_copy = test_dir / "data_v1_copy.csv"
file1_copy.write_text("id,value\n1,100\n2,200\n3,300\n")

hash1 = compute_data_hash(file1)
hash1_copy = compute_data_hash(file1_copy)

print("Property 1: Same content = same hash")
print(f"  {file1.name}: {hash1}")
print(f"  {file1_copy.name}: {hash1_copy}")
assert hash1 == hash1_copy, "Identical content should have identical hash!"
print("  ✓ Hashes match\n")

# Property 2: Different content = different hash
file2 = test_dir / "data_v2.csv"
file2.write_text("id,value\n1,100\n2,200\n3,300\n4,400\n")  # Added row

hash2 = compute_data_hash(file2)
print("Property 2: Different content = different hash")
print(f"  {file1.name}: {hash1}")
print(f"  {file2.name}: {hash2}")
assert hash1 != hash2, "Different content should have different hash!"
print("  ✓ Hashes differ\n")

# Property 3: One byte change = completely different hash (avalanche effect)
file3 = test_dir / "data_v3.csv"
file3.write_text("id,value\n1,101\n2,200\n3,300\n")  # Changed 100 to 101

hash3 = compute_data_hash(file3)
print("Property 3: One byte change = avalanche (completely different hash)")
print(f"  Original: {hash1}")
print(f"  1 byte changed: {hash3}")
assert hash1 != hash3, "Even tiny changes should produce different hash!"
print("  ✓ Cryptographic avalanche confirmed\n")

print("This is exactly how DVC detects data changes:")
print("  - Hash is stored in .dvc file (committed to git)")
print("  - Data is stored separately (not in git)")
print("  - Same hash = same data, guaranteed")

**Key Insight:** Content addressing means the filename doesn't matter—only the bytes do. Two files with different names but identical content get the same ID. This is how git and DVC both work internally.

## Solution 7: Float Comparison Trap

Why `==` breaks and how to fix it.

In [ ]:
import numpy as np

# Demonstrate the problem
a, b, c = 0.1, 0.2, 0.3

left = (a + b) + c
right = a + (b + c)

print("Floating-point arithmetic is NOT associative:")
print(f"  (0.1 + 0.2) + 0.3 = {left:.20f}")
print(f"  0.1 + (0.2 + 0.3) = {right:.20f}")
print(f"  Are they ==? {left == right}")
print(f"  Difference: {abs(left - right):.2e}\n")

# Why this happens
print("Why: Binary floating-point cannot represent 0.1 exactly")
print(f"  0.1 is actually: {0.1:.60f}")
print(f"  0.2 is actually: {0.2:.60f}\n")

# The solution
def safe_compare_predictions(pred1: np.ndarray, pred2: np.ndarray, rtol: float = 1e-9) -> bool:
    """Safely compare two prediction arrays using relative tolerance."""
    return np.allclose(pred1, pred2, rtol=rtol, atol=1e-12)

# Test with nearly-identical arrays
predictions_1 = np.array([0.1 + 0.2, 0.3 + 0.4])
predictions_2 = np.array([0.3, 0.7])

print("Safe comparison in action:")
print(f"  Array 1: {predictions_1}")
print(f"  Array 2: {predictions_2}")
print(f"  Direct == check: {(predictions_1 == predictions_2).all()}")
print(f"  Safe compare: {safe_compare_predictions(predictions_1, predictions_2)}")
print(f"  Difference: {np.abs(predictions_1 - predictions_2)}")

# Real-world example with model predictions
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

X, y = load_iris(return_X_y=True)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X[:100], y[:100])

# Same predictions computed twice
pred1 = model.predict_proba(X[100:])
pred2 = model.predict_proba(X[100:])

print("\nReal model predictions:")
print(f"  Bitwise identical: {(pred1 == pred2).all()}")
print(f"  Numerically close: {safe_compare_predictions(pred1, pred2)}")
print("\n✓ Always use np.allclose for reproducibility tests")

**Best Practice:** 
- Use `np.allclose(a, b, rtol=1e-9)` for reproducibility tests
- Use `np.isclose()` for element-wise comparison
- NEVER use `==` for floating-point arrays in assertions
- On GPU, tolerance needs to be even looser (1e-5 or 1e-6)

## Solution 8: The Full Pipeline

Bringing it all together: reproducible end-to-end training.

In [ ]:
import json
import hashlib
import os
import random
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import joblib
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

@dataclass
class Config:
    """Complete training configuration."""
    n_estimators: int
    max_depth: int
    seed: int
    test_size: float
    data_hash: str  # version ID for the dataset

def set_all_seeds(seed: int) -> np.random.Generator:
    """Seed all random sources."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    return np.random.default_rng(seed)

def compute_data_hash(X: np.ndarray, y: np.ndarray) -> str:
    """Hash the dataset for version tracking."""
    combined = np.concatenate([X.ravel(), y.ravel()])
    return hashlib.sha256(combined.tobytes()).hexdigest()[:12]

def train_pipeline(config: Config) -> tuple:
    """Run complete reproducible training pipeline."""
    # Reset all random sources
    rng = set_all_seeds(config.seed)
    
    # Load data
    X, y = load_breast_cancer(return_X_y=True)
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config.test_size, random_state=config.seed, stratify=y
    )
    
    # Train
    model = RandomForestClassifier(
        n_estimators=config.n_estimators,
        max_depth=config.max_depth,
        random_state=config.seed,
        n_jobs=1  # Deterministic threading
    )
    model.fit(X_train, y_train)
    
    # Evaluate
    predictions = model.predict(X_test)
    score = model.score(X_test, y_test)
    
    return model, predictions, score, X_test, y_test

def save_run(run_dir: Path, config: Config, metrics: dict, model) -> None:
    """Save complete experiment."""
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(asdict(config), indent=2))
    (run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    joblib.dump(model, run_dir / "model.joblib")

# Compute data version
X, y = load_breast_cancer(return_X_y=True)
data_hash = compute_data_hash(X, y)

print(f"Dataset hash: {data_hash}\n")

# Run 1
print("=== RUN 1 ===")
cfg1 = Config(n_estimators=50, max_depth=5, seed=42, test_size=0.2, data_hash=data_hash)
model1, pred1, score1, X_test1, y_test1 = train_pipeline(cfg1)
save_run(Path("sample_runs/full_pipeline_1"), cfg1, {"accuracy": round(score1, 4)}, model1)
print(f"Accuracy: {score1:.4f}")
print(f"First 5 predictions: {pred1[:5]}")

# Run 2 (same config)
print("\n=== RUN 2 (same config) ===")
cfg2 = Config(n_estimators=50, max_depth=5, seed=42, test_size=0.2, data_hash=data_hash)
model2, pred2, score2, X_test2, y_test2 = train_pipeline(cfg2)
save_run(Path("sample_runs/full_pipeline_2"), cfg2, {"accuracy": round(score2, 4)}, model2)
print(f"Accuracy: {score2:.4f}")
print(f"First 5 predictions: {pred2[:5]}")

# Verify perfect reproducibility
print("\n=== REPRODUCIBILITY CHECK ===")
assert cfg1 == cfg2, "Configs differ!"
print("✓ Configs identical")

assert np.array_equal(X_test1, X_test2), "Test sets differ!"
print("✓ Test sets identical")

assert np.array_equal(pred1, pred2), "Predictions differ!"
print("✓ Predictions identical")

assert score1 == score2, "Scores differ!"
print("✓ Scores identical")

print("\n🎉 PERFECT REPRODUCIBILITY ACHIEVED")
print("\nWhat made this possible:")
print("  1. Config dataclass captures all hyperparameters")
print("  2. set_all_seeds() before training")
print("  3. random_state passed to every randomized component")
print("  4. Data versioned with content hash")
print("  5. Everything saved to timestamped directory")

**Expected Output:**
```
Dataset hash: 8f5c9b2a1d3e

=== RUN 1 ===
Accuracy: 0.9650
First 5 predictions: [1 0 0 1 1]

=== RUN 2 (same config) ===
Accuracy: 0.9650
First 5 predictions: [1 0 0 1 1]

=== REPRODUCIBILITY CHECK ===
✓ Configs identical
✓ Test sets identical
✓ Predictions identical
✓ Scores identical

🎉 PERFECT REPRODUCIBILITY ACHIEVED
```

## Solution 9: Environment Snapshot

Capturing the execution environment.

In [ ]:
from importlib.metadata import version, PackageNotFoundError
import json
import sys
from pathlib import Path
from datetime import datetime

def capture_environment() -> dict:
    """Capture versions of key ML packages and Python version."""
    packages = ["numpy", "pandas", "scikit-learn", "matplotlib", "joblib", "scipy"]
    
    env = {
        "python_version": sys.version,
        "captured_at": datetime.now().isoformat(),
        "packages": {}
    }
    
    for pkg in packages:
        try:
            env["packages"][pkg] = version(pkg)
        except PackageNotFoundError:
            env["packages"][pkg] = "NOT_INSTALLED"
    
    return env

# Capture and save
env = capture_environment()
snapshot_file = Path("sample_data/environment_snapshot.json")
snapshot_file.parent.mkdir(exist_ok=True)
snapshot_file.write_text(json.dumps(env, indent=2))

# Display
print("Current environment snapshot:")
print("=" * 50)
print(f"Python: {env['python_version'].split()[0]}")
print(f"Captured: {env['captured_at']}")
print("\nPackages:")
for pkg, ver in env["packages"].items():
    print(f"  {pkg:15} {ver}")

print(f"\n✓ Saved to {snapshot_file}")
print("\nCommit this alongside experimental results for full auditability.")

**Best Practice:** Capture environment at the *start* of each experiment. When results differ unexpectedly, the environment snapshot is your first debugging clue.

## Solution 10: Debugging Irreproducibility (Challenge)

Three bugs, all related to incomplete seeding.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_digits

# BUGS:
# Bug 1: train_test_split missing random_state parameter
# Bug 2: RandomForestClassifier missing random_state parameter  
# Bug 3: The shuffle uses np.random but doesn't control its seed per-call

def fixed_training_pipeline(seed=42):
    """Fully reproducible version with all bugs fixed."""
    # FIX: Reset numpy seed at the START of each call
    np.random.seed(seed)
    
    X, y = load_digits(return_X_y=True)
    
    # FIX 1: Add random_state to train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=True, random_state=seed
    )
    
    # Shuffle training data (now reproducible because np.random.seed was called)
    shuffle_idx = np.random.permutation(len(X_train))
    X_train = X_train[shuffle_idx]
    y_train = y_train[shuffle_idx]
    
    # FIX 2: Add random_state to RandomForestClassifier
    model = RandomForestClassifier(
        n_estimators=20, max_depth=10, random_state=seed
    )
    model.fit(X_train, y_train)
    
    return model.predict(X_test)

# Verify reproducibility
print("Testing fixed version...")
pred1 = fixed_training_pipeline(seed=7)
pred2 = fixed_training_pipeline(seed=7)

print(f"First 10 predictions (run 1): {pred1[:10]}")
print(f"First 10 predictions (run 2): {pred2[:10]}")
print(f"\nPredictions match: {(pred1 == pred2).all()}")
assert (pred1 == pred2).all(), "Still not reproducible!"
print("\n✓ ALL BUGS FIXED - training is now fully reproducible")

# Educational summary
print("\n" + "="*60)
print("LESSONS LEARNED:")
print("="*60)
print("1. train_test_split MUST receive random_state parameter")
print("2. All sklearn estimators MUST receive random_state parameter")
print("3. Call np.random.seed() at the START of each training function")
print("4. Any operation using np.random.* inherits the global seed")
print("\nGeneral rule: Grep your code for 'random' and ensure EVERY")
print("              source is either seeded or receives random_state")

**Debugging Strategy:**
1. Run twice, compare outputs
2. Binary search: comment out half the code, find which half differs
3. Check every function that could use randomness
4. Verify each has explicit `random_state` parameter set

## Solution 11: PYTHONHASHSEED Mystery

Understanding process-level environment variables.

In [ ]:
# EXPLANATION:

# Why it doesn't work:
# PYTHONHASHSEED is read by the Python interpreter at STARTUP, before any
# Python code runs. It's used to initialize the hash secret for the entire
# process. Setting os.environ["PYTHONHASHSEED"] inside a running script
# documents your intent but cannot change the already-initialized hash secret.
# The hash secret is baked into the C layer and cannot be changed after
# interpreter initialization.

# What to do instead:
# Option 1: Export before running Python
#   $ PYTHONHASHSEED=42 python train.py
#   or on Windows:
#   > set PYTHONHASHSEED=42
#   > python train.py
#
# Option 2: Use subprocess to launch a child process with the variable set
#   import os, subprocess
#   env = os.environ.copy()
#   env["PYTHONHASHSEED"] = "42"
#   subprocess.run(["python", "train.py"], env=env)

# When this matters in ML:
# - When you iterate over sets or dicts and their order affects results
# - Feature names in dict-based pipelines (different order = different encoding)
# - Column selection where duplicate names exist (first one seen wins)
# - Any algorithm that breaks ties by order of encounter
# - Rare in pure NumPy/sklearn, more common in pandas/dict-heavy code

# Demonstration:
import os

# This does NOTHING for the current process
os.environ["PYTHONHASHSEED"] = "42"

# Set iteration order is still randomized
s = {"apple", "banana", "cherry"}
print("Set iteration (attempt 1):", list(s))

# Creating a fresh set doesn't help - the hash secret is already fixed
s2 = {"apple", "banana", "cherry"}
print("Set iteration (attempt 2):", list(s2))

print("\n⚠️  Both use the same (random) hash secret from process startup")
print("Setting os.environ after launch has no effect.")
print("\nCorrect approach: PYTHONHASHSEED=42 python script.py")

## Solution 12: Experiment Comparison (Challenge)

Building a leaderboard from run directories.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
from dataclasses import dataclass, asdict
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib

def compare_runs(run_dirs: list[Path], sort_by: str = "accuracy", ascending: bool = False) -> pd.DataFrame:
    """Load and compare multiple experimental runs."""
    records = []
    
    for run_dir in run_dirs:
        run_dir = Path(run_dir)
        if not run_dir.exists():
            continue
            
        # Load config and metrics
        config = json.loads((run_dir / "config.json").read_text())
        metrics = json.loads((run_dir / "metrics.json").read_text())
        
        # Flatten into single record
        record = {"run_name": run_dir.name}
        record.update({f"config_{k}": v for k, v in config.items()})
        record.update(metrics)
        records.append(record)
    
    df = pd.DataFrame(records)
    
    # Sort by specified metric
    if sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=ascending)
    
    return df

# Create multiple runs with different configs
@dataclass
class ExperimentConfig:
    n_estimators: int
    max_depth: int
    seed: int

def run_experiment(name: str, config: ExperimentConfig) -> Path:
    """Run one experiment and save results."""
    np.random.seed(config.seed)
    X, y = load_iris(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=config.seed
    )
    
    model = RandomForestClassifier(
        n_estimators=config.n_estimators,
        max_depth=config.max_depth,
        random_state=config.seed
    )
    model.fit(X_train, y_train)
    
    accuracy = model.score(X_test, y_test)
    
    # Save
    run_dir = Path(f"sample_runs/comparison/{name}")
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(asdict(config), indent=2))
    (run_dir / "metrics.json").write_text(json.dumps({"accuracy": round(accuracy, 4)}, indent=2))
    joblib.dump(model, run_dir / "model.joblib")
    
    return run_dir

# Run multiple experiments
print("Running experiments...\n")
experiments = [
    ("baseline", ExperimentConfig(n_estimators=10, max_depth=3, seed=42)),
    ("more_trees", ExperimentConfig(n_estimators=50, max_depth=3, seed=42)),
    ("deeper", ExperimentConfig(n_estimators=10, max_depth=10, seed=42)),
    ("best", ExperimentConfig(n_estimators=50, max_depth=10, seed=42)),
]

run_paths = []
for name, config in experiments:
    path = run_experiment(name, config)
    run_paths.append(path)
    print(f"✓ Completed: {name}")

# Compare all runs
print("\n" + "="*70)
print("EXPERIMENT LEADERBOARD")
print("="*70)
df = compare_runs(run_paths, sort_by="accuracy", ascending=False)

# Display key columns
display_cols = ["run_name", "config_n_estimators", "config_max_depth", "accuracy"]
print(df[display_cols].to_string(index=False))

print("\n" + "="*70)
winner = df.iloc[0]
print(f"🏆 WINNER: {winner['run_name']}")
print(f"   Accuracy: {winner['accuracy']:.4f}")
print(f"   Config: n_estimators={winner['config_n_estimators']}, max_depth={winner['config_max_depth']}")

# Show how to load the winning model
print("\nTo load the winning model:")
winner_path = Path(f"sample_runs/comparison/{winner['run_name']}")
print(f"  model = joblib.load('{winner_path / 'model.joblib'}')")

**Expected Output:**
```
Running experiments...

✓ Completed: baseline
✓ Completed: more_trees
✓ Completed: deeper
✓ Completed: best

======================================================================
EXPERIMENT LEADERBOARD
======================================================================
run_name     config_n_estimators  config_max_depth  accuracy
best                          50                10    0.9667
more_trees                    50                 3    0.9667
deeper                        10                10    0.9333
baseline                      10                 3    0.9333

======================================================================
🏆 WINNER: best
   Accuracy: 0.9667
   Config: n_estimators=50, max_depth=10
```

**Best Practice:** This pattern scales to hundreds of experiments. The filesystem IS your database—no special tooling required.